In [ ]:
from pathlib import Path # Python's standard library module for working with filesystem paths in an object-oriented way
from openai import OpenAI 
from dotenv import load_dotenv # Load environment variable
from pydantic import BaseModel, Field # For Domain models
from chromadb import PersistentClient # Chroma database
from tqdm import tqdm

### Package - Litellm 

The litellm library is a Python package that gives you a unified interface for calling many different large language model (LLM) providers — OpenAI, Anthropic, Azure, Cohere, Hugging Face, Google's Gemini/Vertex AI, AWS Bedrock, Ollama, and dozens more — all through the same function signature.

### Function - completion() 

The completion() function specifically mimics the OpenAI ChatCompletion API format. So instead of learning each provider's own SDK, you call:

```bash
response = completion(
    model="gpt-4",  # or "claude-sonnet-4-6", "gemini/gemini-pro", etc.
    messages=[{"role": "user", "content": "Hello!"}]  
)
```


In [ ]:
from litellm import completion
import numpy as np

### TSNE 
TSNE is a class in scikit-learn's sklearn.TSNE is a class in scikit-learn's sklearn.manifold module, used for t-distributed Stochastic Neighbor Embedding — a technique for dimensionality reduction, mainly used for visualizing high-dimensional data.

In [ ]:
from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [ ]:
# Load environment variables
load_dotenv(override=True)

# Model name
MODEL = "gpt-4.1-nano"

# Vector database and related details
DB_NAME = "preprocessed_db"
collection_name = "docs"
embedding_model = "text-embedding-3-large"

# Knowledge source path
KNOWLEDGE_BASE_PATH = Path("knowledge-base")
AVERAGE_CHUNK_SIZE = 500

In [ ]:
# Get a handle of OpenAI object
openai = OpenAI()

In [ ]:
class Result(BaseModel):
    page_content: str
    metadata : dict

In [ ]:
# Class to represent a Chunk
class Chunk (BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    originial_text : str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    '''
    document is a dict
    metadata is a dict
    '''

    def as_result(self, document):
        metadata = { 
            "source" : document["source"],
            "type" : document["type"]
        }
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.originial_text, metadata=metadata)

class Chunks (BaseModel):
    chunks : list[Chunk]